# Wage prediction

[Causal ML book](https://causalml-book.org/chapters/CausalML_chap_1.pdf)

[Collab notebook](https://colab.research.google.com/github/CausalAIBook/MetricsMLNotebooks/blob/main/PM1/python-ols-and-lasso-for-wage-prediction.ipynb#scrollTo=qb_bGht76jYb)

In [ ]:
import numpy as np
import pandas as pd

from tabulate import tabulate
import textwrap

In [ ]:
file = "https://raw.githubusercontent.com/CausalAIBook/MetricsMLNotebooks/main/data/wage2015_subsample_inference.csv"
df = pd.read_csv(file)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
# matching table 1.1 in the example
df[['lwage', 'shs', 'hsg', 'scl', 'clg', 'ad', 'we']].mean()

In [ ]:
# make table 1.1 look nicer
print_df = df[['lwage', 'shs', 'hsg', 'scl', 'clg', 'ad', 'we']].mean()
print(tabulate(print_df.to_frame(name='Sample Mean'), floatfmt='.4f', headers='keys'))

In [ ]:
# same table by gender
avg_df = df.groupby('sex')[['lwage', 'shs', 'hsg', 'scl', 'clg', 'ad', 'we']].mean()
# transpose to make it look nice
avg_df = avg_df.T
# replace the headers with 0 = male, 1 = female
avg_df = avg_df.rename(columns={0: 'male', 1: 'female'})
print(tabulate(avg_df, floatfmt='.4f'))

In [ ]:
diff_df = print_df.diff().loc[1]
# 
diff_df = pd.concat([
    avg_df,
    diff_df.to_frame(name='diff')
], axis=1)
print(tabulate(diff_df, floatfmt='.4f', headers='keys'))

print(f"\nIn this sample, women earn {diff_df.loc['lwage', 'diff']:.1%} less than men.")

## Prediction

In [ ]:
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.20, random_state=123)

In [ ]:
ols_mod_base = smf.ols('lwage ~ sex + exp1 + shs + hsg+ scl + clg + mw + so + we + C(occ2) + C(ind2)', data=train_df)
ols_base_result = ols_mod_base.fit()

ols_base_pred = ols_base_result.predict(train_df)
# ols_base_result.summary()


In [ ]:
check_cols = ['sex', 'exp1', 'shs', 'hsg', 'scl', 'clg', 'mw', 'so', 'we']
ols_base_result.summary2().tables[1].loc[check_cols]

In [ ]:
print_str = (
    f"Women in the sample earn {ols_base_result.params['sex']:.4f} less than males,"
    " controlling for job characteristics."
)
print(textwrap.fill(print_str, 80))

**Aside: partialing out**

In [ ]:
y_partial = smf.ols('lwage ~ exp1 + shs + hsg+ scl + clg + mw + so + we + C(occ2) + C(ind2)', data=train_df)
y_partial_result = y_partial.fit()

d_partial = smf.ols('sex ~ exp1 + shs + hsg+ scl + clg + mw + so + we + C(occ2) + C(ind2)', data=train_df)
d_partial_result = d_partial.fit()

In [ ]:
reg_dict = {
    'y_tilde': y_partial_result.resid, 
    'd_tilde': d_partial_result.resid
}
base_ols_fwl = smf.ols('y_tilde ~ d_tilde', data=reg_dict)
base_ols_fwl_result = base_ols_fwl.fit()

base_ols_fwl_result.summary()

In [ ]:
def analysis_variance(mod_fit):
    r_squared = mod_fit.rsquared
    rsquared_adj = mod_fit.rsquared_adj
    mse = mse_base = np.mean(mod_fit.resid**2)
    mse_adj = mod_fit.mse_resid

    return pd.Series({
        'r_squared': r_squared,
        'rsquared_adj': rsquared_adj,
        'mse': mse,
        'mse_adj': mse_adj
    })

analysis_variance(ols_fit)

In [ ]:
ols_base_result.predict(test_df)
